In [1]:
import torch
import cv2
from pathlib import Path
from loguru import logger
import pandas as pd
import os
import argparse
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import tqdm
import sys


class DataSet(torch.utils.data.Dataset):
    def __init__(
        self,
        input_dir: Path,
        labels: list[str],
        data_type: str,
        adversarial_type: str,
        max_samples: int = -1000,
    ):

        self.input_dir = input_dir
        self.labels = labels
        self.data_type = data_type
        self.adversarial_type = adversarial_type
        self.npz_files = list(input_dir.glob(f"{adversarial_type}/{data_type}/*.npz"))
        self.max_samples = max_samples

    def __len__(self):
        return (
            len(self.npz_files)
            if self.max_samples < 0
            else min(len(self.npz_files), self.max_samples)
        )

    def __getitem__(self, idx):
        selected_file = self.npz_files[idx]
        npz_data = np.load(selected_file)
        input_image = npz_data["inputs"]
        adversarial_image = npz_data["adversarial"]
        label = npz_data["label_str"].item()

        label_index = self.labels.index(label)
        img_tensor = (
            torch.tensor(input_image, dtype=torch.float32, device=device).unsqueeze(0)
            / 255
        )
        adv_tensor = (
            torch.tensor(
                adversarial_image, dtype=torch.float32, device=device
            ).unsqueeze(0)
            / 255
        )
        self.curr_label = label

        return img_tensor, adv_tensor, label_index


ModuleNotFoundError: No module named 'tqdm'

In [ ]:


parser = argparse.ArgumentParser(
    description="Evaluate adversarial examples against a classifier and an autoencoder."
)
parser.add_argument(
    "--data_dir",
    type=str,
    required=True,
    help="Directory containing the input images.",
)
parser.add_argument(
    "--batch_size",
    type=int,
    default=32,
    help="Batch size for processing images.",
)
parser.add_argument(
    "--output_dir",
    type=str,
    default="/home/hpc/iwi7/iwi7101h/i7-IDS/results/adversarial_blocking",
    help="Directory to save the output results.",
)
parser.add_argument(
    "--results_dir",
    type=str,
    default="/home/hpc/iwi7/iwi7101h/i7-IDS/results",
    help="Directory containing the project structure with models.",
)
parser.add_argument(
    "--data_type",
    type=str,
    default="normal",
    choices=["normal", "normalized"],
)
parser.add_argument(
    "--max_samples",
    type=int,
    default=-1000,
    help="Maximum number of samples to process. Use -1000 for all samples.",
)
parser.add_argument("--data_split", type=str, default="val")

args = parser.parse_args()
data_dir = Path(args.data_dir)
output_dir = Path(args.output_dir)
data_split = args.data_split
data_type = args.data_type
batch_size = args.batch_size

if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)
# Set the project directory
results_dir = Path(args.results_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

adversarial_names = [a.name for a in data_dir.iterdir()]

logger.info(f"Found {len(adversarial_names)} adversarial types.")
logger.info(f"\n{adversarial_names}")


def get_predictions(
    clf_model,
    adv_model,
    ae_model,
    input_batch,
    adversarial_batch,
):
    input_tensor = torch.tensor(
        np.array(input_batch), dtype=torch.float32, device=device
    ).unsqueeze(1)
    adversarial_tensor = torch.tensor(
        np.array(adversarial_batch),
        dtype=torch.float32,
        device=device,
    ).unsqueeze(1)

    with torch.no_grad():
        # clean pred
        clf_preds, proba = clf_model(input_tensor)
        # adv pred
        adv_preds, proba = adv_model(adversarial_tensor)
        # reconstructed
        recon_tensor = ae_model(input_tensor)
        # blocked preds
        blocked_preds = clf_model(recon_tensor)
        # adversarial blocked preds
        adv_blocked_preds = adv_model(recon_tensor)

        mae = torch.mean(torch.abs(adversarial_tensor - recon_tensor)).item()

    clf_preds = clf_preds.cpu().numpy()
    adv_preds = adv_preds.cpu().numpy()
    recon_tensor = recon_tensor.cpu().numpy()
    blocked_preds = blocked_preds.cpu().numpy()
    adv_blocked_preds = adv_blocked_preds.cpu().numpy()
    return (clf_preds, adv_preds, recon_tensor, blocked_preds, adv_blocked_preds, mae)


# autoencoder_models = [
#     Path("autoencoder/rdunet_normal"),
#     Path("autoencoder/rdunet_normalized"),
#     Path("autoencoder/unet_custom_normal"),
#     Path("autoencoder/unet_custom_normalized"),
# ]

labels = [
    "REPLAY",
    "DNP3_INFO",
    "DNP3_ENUMERATE",
    "STOP_APP",
    "NORMAL",
    "INIT_DATA",
    "COLD_RESTART",
    "WARM_RESTART",
    "DISABLE_UNSOLICITED",
]

clf_adv_ae_models = [
    [
        Path("image_classification/resnet18_normal_nosampling"),
        Path("adversarial/resnet18_normal_nosampling_adv"),
        Path("autoencoder/rdunet_normal"),
    ],
    [
        Path("image_classification/resnet18_normalized_nosampling"),
        Path("adversarial/resnet18_normalized_nosampling_adv"),
        Path("autoencoder/rdunet_normalized"),
    ],
    [
        Path("image_classification/mobilenet_v3_large_normal_nosampling"),
        Path("adversarial/mobilenet_v3_large_normal_nosampling_adv"),
        Path("autoencoder/rdunet_normal"),
    ],
    [
        Path("image_classification/mobilenet_v3_large_normalized_nosampling"),
        Path("adversarial/mobilenet_v3_large_normalized_nosampling_adv"),
        Path("autoencoder/rdunet_normalized"),
    ],
    [
        Path("image_classification/resnet18_custom_normal_nosampling"),
        Path("adversarial/resnet18_custom_normal_nosampling_adv"),
        Path("autoencoder/unet_custom_normal"),
    ],
    [
        Path("image_classification/resnet18_custom_normalized_nosampling"),
        Path("adversarial/resnet18_custom_normalized_nosampling_adv"),
        Path("autoencoder/unet_custom_normalized"),
    ],
    [
        Path("image_classification/mobilenet_v3_large_custom_normal_nosampling"),
        Path("adversarial/mobilenet_v3_large_custom_normal_nosampling_adv"),
        Path("autoencoder/unet_custom_normal"),
    ],
    [
        Path("image_classification/mobilenet_v3_large_custom_normalized_nosampling"),
        Path("adversarial/mobilenet_v3_large_custom_normalized_nosampling_adv"),
        Path("autoencoder/unet_custom_normalized"),
    ],
]

completed_non_adversarial = False


results = []
for adversarial_name in adversarial_names:

    logger.info(f"Processing adversarial type: {adversarial_name}")
